# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [2]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [3]:
# 创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

for epoch in range(1, 2001):
    # 设置本轮学习率，并清空旧梯度
    for param_group in optimizer.param_groups:
        param_group['lr'] = 0.1 / (epoch ** 0.5)
    optimizer.zero_grad()

    # 调用 predict，同时使用 x1 和 x2，得到 shape 为 (6,) 的预测
    y_pred = predict(x1, x2, w1, w2, b)

    # 逐样本绝对误差 (6,)，再取平均得到标量损失 ()
    per_sample_loss = torch.abs(y - y_pred)
    loss = per_sample_loss.mean()

    # 对平均损失反向传播一次，更新全部参数
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f'epoch {epoch:4d}  lr = {current_lr:.4f}  loss = {loss.item():.6f}')

# 在 torch.no_grad() 中，用训练后的参数重新计算预测与损失（不再记录梯度）
with torch.no_grad():
    y_pred = predict(x1, x2, w1, w2, b)
    per_sample_loss = torch.abs(y - y_pred)
    loss = per_sample_loss.mean()

print('w1 =', w1.item(), ' w2 =', w2.item(), ' b =', b.item())
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))
print('y_pred:', y_pred, 'shape:', tuple(y_pred.shape))
print('逐样本绝对误差:', per_sample_loss, 'shape:', tuple(per_sample_loss.shape))
print('平均损失 loss:', loss, 'shape:', tuple(loss.shape))


epoch  500  lr = 0.0045  loss = 0.245472
epoch 1000  lr = 0.0032  loss = 0.003635


epoch 1500  lr = 0.0026  loss = 0.002817
epoch 2000  lr = 0.0022  loss = 0.003900
w1 = 1.999006986618042  w2 = -3.0010085105895996  b = 0.9993674159049988
x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)
y_pred: tensor([-5.9997, -0.9996, -5.0026,  5.9994, -4.0056,  3.9954]) shape: (6,)
逐样本绝对误差: tensor([0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046]) shape: (6,)
平均损失 loss: tensor(0.0024) shape: ()


## 我的观察与实验

**1. 一次预测需要几个输入张量？x1、x2 与样本数 6 的关系（10 分）**

一次预测需要**两个输入张量** `x1` 和 `x2`（另外还要用到三个可学习参数 `w1`、`w2`、`b`）。`x1`、`x2` 的 shape 都是 `(6,)`：第 i 个元素属于第 i 个样本，6 是**样本数**，不是特征数；两个特征被拆到两个张量上，按样本下标一一对齐。`predict` 一次调用返回 `(6,)`，其中第 i 个预测只由 `x1[i]`、`x2[i]` 决定，所以“一个样本两个输入”和“共 6 个样本”是两个维度，互不冲突。

**2. 对照上周：改了哪里，哪里没变（10 分）**

| 部分 | 上周（单一输入） | 本周（多输入） |
|---|---|---|
| 预测公式 | `w * x + b` | `w1 * x1 + w2 * x2 + b`，多一个输入项 |
| 参数列表 | 1 个 `w` | 3 个标量 `w1`、`w2`、`b`，都要 `requires_grad=True` 并一起放进优化器 |
| 损失计算 | 单个样本的 `abs(y - y_pred)`，结果是标量 | 先算 `(6,)` 的逐样本误差，再用 `.mean()` 压成 `()` 的平均损失（标量才能 `backward`） |
| 优化器步骤 | `zero_grad` → 前向 → 损失 → `backward` → `step` | 完全不变，只是每轮多改了学习率、参数组里多了两个参数 |

**3. 最终参数、预测与平均损失，如何判断训练有效（20 分）**

- 最终参数：`w1 ≈ 2.000`、`w2 ≈ -3.000`、`b ≈ 1.000`。
- 预测：`y_pred ≈ [-6.00, -1.00, -5.00, 6.00, -4.00, 4.00]`，与 `y = [-6, -1, -5, 6, -4, 4]` 逐项对应。
- 逐样本绝对误差：全部小于 0.006；平均损失 `loss ≈ 0.0024`（初始值 `mean|y| = (6+1+5+6+4+4)/6 = 26/6 ≈ 4.333`）。
- 训练轨迹：500 轮 `loss = 0.245`、1000 轮 `0.0036`、1500 轮 `0.0028`、2000 轮 `0.0039`。前 1000 轮把损失降了约三个数量级，之后停在千分位并小幅波动——绝对值损失在残差接近 0 时梯度恒为 `±sign(y - y_pred)*x`，学习率又始终不为 0，参数会在最优点附近来回穿越，所以末段并非严格单调。这提醒我们判断收敛要看“是否稳定在很小的量级”，不能只盯最后一轮的单个数字。
- 判断训练有效的依据（三条同时满足才算有效，不能只看参数大小）：
  1. 损失从初值 4.333 下降约三个数量级后稳定在千分位，没有震荡放大或发散；
  2. 残差远小于目标量级：平均损失 0.0024 相对 4.333 已降到千分之一以下，逐样本预测与目标几乎重合；
  3. 学到的参数能解释数据本身：把 `w1=2, w2=-3, b=1` 代回任一行都严格满足 `y = 2*x1 - 3*x2 + 1`（例如第 4 个样本 `2*2 - 3*3 + 1 = -4`），说明模型拟合出的是数据里的真实线性关系，而不是碰巧把平均值拉低。

**4. 为什么 w1=w2=b=0 仍然可以开始学习（10 分）**

因为本题的预测是纯线性的，零初始化时损失对参数的**梯度不为 0**。此时 `y_pred` 全为 0，`L = mean|y - 0|`，对 `w1` 的梯度是 `mean(-sign(y) * x1) = -0.833`（`w2` 为 `1.0`，`b` 为 `0.333`），第一次 `step` 就把三个参数推出了原点，随后每轮继续得到新梯度。这里没有“参数为 0 就永远不动”的问题：上周那个参数被平方的例子里，`w=0` 时输出恒为 0 且梯度也被压成 0，参数会停在原点；而线性形式在 0 点处梯度仍然由数据决定。真正的风险是对称性——如果所有输入、所有参数都完全相同，梯度会一样，参数就同步更新、无法区分；本周 `x1`、`x2` 取值不同，两个权重各自拿到不同梯度，所以零初始化完全可以正常工作。
